In [1]:
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Optional
from transformers import Qwen2ForCausalLM
from transformers.modeling_outputs import CausalLMOutputWithPast
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

/Users/simone/Projects/university/titans/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────
 
@dataclass
class TitansConfig:
    chunk_size: int = 64            # C — segment length
    n_persistent: int = 16          # N_p — persistent memory tokens
    memory_depth: int = 2           # L_M — MLP depth (>=2 = "deep memory")
    use_linear_memory: bool = True  # True=Delta Rule (fast), False=MLP (expressive)
    inner_lr: float = 0.01          # θ — inner-loop learning rate scale
    freeze_base: bool = True        # freeze Qwen2 weights (set False to fine-tune all)

In [3]:
# ─────────────────────────────────────────────
# Neural Memory — Linear variant (Delta Rule + momentum + weight decay)
# Fast, parallelizable, O(D²) memory per layer.
# ─────────────────────────────────────────────

class NeuralMemoryLinear(nn.Module):
    """
    Matrix-valued memory M ∈ ℝ^{D×D}.
    Update rule (generalized Gated DeltaNet with momentum):
        S_t = η_t · S_{t-1} − θ_t · (Mk − v)k^T    # momentum surprise
        M_t = (1−α_t) · M_{t-1} + S_t               # forget + accumulate
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.d = d_model
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        # Data-dependent gates (all ∈ (0,1) via sigmoid)
        self.alpha_proj = nn.Linear(d_model, 1)   # forget
        self.theta_proj = nn.Linear(d_model, 1)   # lr scale
        self.eta_proj   = nn.Linear(d_model, 1)   # momentum decay
        # Buffers reset per sequence
        self.M: Optional[torch.Tensor] = None
        self.S: Optional[torch.Tensor] = None
 
    def reset(self, B: int, device: torch.device, dtype: torch.dtype = torch.bfloat16):
        self.M = torch.zeros(B, self.d, self.d, device=device, dtype=dtype)
        self.S = torch.zeros_like(self.M)
 
    def retrieve(self, x: torch.Tensor) -> torch.Tensor:
        """q → M*q  (read without updating weights)."""
        q = F.normalize(self.W_Q(x), dim=-1)           # (B,T,D)
        return torch.bmm(q, self.M.transpose(-1, -2))  # (B,T,D)
 
    @torch.no_grad()
    def update(self, x: torch.Tensor):
        """Surprise-based write (in-place, no autograd)."""
        B, T, D = x.shape
        k = F.normalize(self.W_K(x), dim=-1)  # (B,T,D)
        v = self.W_V(x)
 
        avg = x.mean(1, keepdim=True)          # (B,1,D) for gates
        alpha = torch.sigmoid(self.alpha_proj(avg)).squeeze(-1)  # (B,1)
        theta = torch.sigmoid(self.theta_proj(avg)).squeeze(-1) * 0.1
        eta   = torch.sigmoid(self.eta_proj(avg)).squeeze(-1)
 
        # Momentary surprise: ∇ℓ = (Mk − v)k^T
        pred  = torch.bmm(k, self.M.transpose(-1, -2))     # (B,T,D)
        error = pred - v                                     # (B,T,D)
        grad_M = torch.bmm(error.transpose(1, 2), k) / T   # (B,D,D)
 
        a = alpha.unsqueeze(-1)
        t = theta.unsqueeze(-1)
        e = eta.unsqueeze(-1)
 
        self.S = e * self.S - t * grad_M
        self.M = (1 - a) * self.M + self.S

In [4]:
# ─────────────────────────────────────────────
# Neural Memory — MLP variant (requires torch.func)
# Deep, non-linear memory. More expressive, slower.
# ─────────────────────────────────────────────
 
class NeuralMemoryMLP(nn.Module):
    """
    MLP-valued memory updated via inner-loop gradient descent.
    Loss: ℓ(M; x) = ‖M(k) − v‖²
    Requires PyTorch >= 2.0 (torch.func).
    """
    def __init__(self, d_model: int, n_layers: int = 2, inner_lr: float = 0.01):
        super().__init__()
        self.inner_lr = inner_lr
        layers: list[nn.Module] = []
        for i in range(n_layers):
            layers.append(nn.Linear(d_model, d_model))
            if i < n_layers - 1:
                layers.append(nn.SiLU())
        self.mlp = nn.Sequential(*layers)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.alpha_proj = nn.Linear(d_model, 1)
        self._fast: dict[str, torch.Tensor] = {}
 
    def reset(self, B: int, device: torch.device, dtype: torch.dtype = torch.bfloat16):
        self._fast = {n: p.detach().clone().to(dtype) for n, p in self.mlp.named_parameters()}
 
    def _fwd(self, x: torch.Tensor, params: dict) -> torch.Tensor:
        from torch.func import functional_call
        return functional_call(self.mlp, params, x)
 
    def retrieve(self, x: torch.Tensor) -> torch.Tensor:
        q = F.normalize(self.W_Q(x), dim=-1)
        if self._fast:
            try:
                return self._fwd(q, self._fast)
            except Exception:
                pass
        return self.mlp(q)
 
    def update(self, x: torch.Tensor):
        k = F.normalize(self.W_K(x), dim=-1)
        v = self.W_V(x)
        alpha = torch.sigmoid(self.alpha_proj(x.mean(1))).item()
        theta = self.inner_lr
 
        try:
            from torch.func import grad, functional_call
 
            def loss_fn(params):
                return F.mse_loss(functional_call(self.mlp, params, k), v)
 
            grads = grad(loss_fn)(self._fast)
            with torch.no_grad():
                self._fast = {
                    n: (1 - alpha) * p - theta * grads[n]
                    for n, p in self._fast.items()
                }
        except ImportError:
            # Fallback: manual SGD (no gradient flow through memory update)
            pred = self.mlp(k)
            loss = F.mse_loss(pred, v)
            loss.backward(retain_graph=True)
            with torch.no_grad():
                for n, p in self.mlp.named_parameters():
                    if p.grad is not None:
                        self._fast[n] = (1 - alpha) * self._fast[n] - theta * p.grad
                        p.grad = None
 

In [5]:
# ─────────────────────────────────────────────
# Persistent Memory
# ─────────────────────────────────────────────

class PersistentMemory(nn.Module):
    """
    Learnable but data-independent tokens prepended to each chunk.
    Encodes task-level knowledge. Fixed at test time.
    """
    def __init__(self, d_model: int, n_tokens: int):
        super().__init__()
        self.tokens = nn.Parameter(torch.randn(1, n_tokens, d_model) * 0.02)
 
    def forward(self, B: int) -> torch.Tensor:
        return self.tokens.expand(B, -1, -1)  # (B, N_p, D)

In [6]:
# ─────────────────────────────────────────────
# Qwen2 + Titans MAC
# ─────────────────────────────────────────────
 
class Qwen2TitansMAC(nn.Module):
    """
    Drop-in wrapper: same input/output signature as Qwen2ForCausalLM.
    New trainable params: neural_memory, persistent_memory, out_gate, conv.
    """
    def __init__(self, base: Qwen2ForCausalLM, cfg: TitansConfig):
        super().__init__()
        self.cfg  = cfg
        self.base = base
        D = base.config.hidden_size  # 1536 for Qwen2-1.5B
 
        if cfg.freeze_base:
            for p in base.parameters():
                p.requires_grad_(False)
 
        # Memory modules
        self.neural_memory: NeuralMemoryLinear | NeuralMemoryMLP
        if cfg.use_linear_memory:
            self.neural_memory = NeuralMemoryLinear(D)
        else:
            self.neural_memory = NeuralMemoryMLP(D, cfg.memory_depth, cfg.inner_lr)
 
        self.persistent = PersistentMemory(D, cfg.n_persistent)
 
        # Output gate: o_t = y_t ⊗ σ(gate(norm(M*(y_t))))
        self.out_norm = nn.RMSNorm(D, eps=1e-6)
        self.out_gate = nn.Linear(D, D, bias=False)
 
        # Depthwise conv on embeddings (§4.4)
        self.conv = nn.Conv1d(D, D, kernel_size=3, padding=1, groups=D, bias=False)
 
    # ── internal helpers ──────────────────────
 
    def _embed(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.base.model.embed_tokens(input_ids)
        # depthwise conv (operates on sequence dim)
        x = x + self.conv(x.transpose(1, 2)).transpose(1, 2)
        return x
 
    def _qwen2_layers(
        self,
        h: torch.Tensor,
        pos: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Run decoder stack + norm on already-embedded hidden states."""
        B, S, _ = h.shape
        if pos is None:
            pos = torch.arange(S, device=h.device).unsqueeze(0).expand(B, -1)
        out = self.base.model(
            inputs_embeds=h,
            attention_mask=None,
            position_ids=pos,
            use_cache=False,
            output_hidden_states=False,
            return_dict=True,
        )
        return out.last_hidden_state
 
    # ── forward ──────────────────────────────
 
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> CausalLMOutputWithPast:
        B, N   = input_ids.shape
        C, Np  = self.cfg.chunk_size, self.cfg.n_persistent
        device = input_ids.device
 
        embeds = self._embed(input_ids)              # (B, N, D)
        self.neural_memory.reset(B, device, dtype=embeds.dtype)
 
        chunks_out: list[torch.Tensor] = []
 
        for i in range((N + C - 1) // C):
            s, e  = i * C, min((i + 1) * C, N)
            chunk = embeds[:, s:e]                   # (B, C', D)
            C_    = e - s
 
            # 1. Retrieve: h_t = M*(q_t)
            h_t = self.neural_memory.retrieve(chunk) # (B, C', D)
 
            # 2. Persistent tokens
            p   = self.persistent(B)                 # (B, Np, D)
 
            # 3. [P || h_t || S(t)]
            x_tilde = torch.cat([p, h_t, chunk], 1) # (B, Np+2C', D)
 
            # 4. Qwen2 attention over extended context
            S_total = x_tilde.shape[1]
            pos = torch.arange(S_total, device=device).unsqueeze(0).expand(B, -1)
            y_all = self._qwen2_layers(x_tilde, pos)  # (B, Np+2C', D)
 
            # Extract S(t) portion only
            y_t = y_all[:, Np + C_ : Np + 2 * C_]   # (B, C', D)
 
            # 5. Update neural memory with attention output
            self.neural_memory.update(y_t)
 
            # 6. Gate: o_t = y_t ⊗ σ(gate(norm(M*(y_t))))
            m_out  = self.neural_memory.retrieve(y_t)
            gate   = torch.sigmoid(self.out_gate(self.out_norm(m_out)))
            o_t    = y_t * gate                       # (B, C', D)
 
            chunks_out.append(o_t)
 
        hidden = torch.cat(chunks_out, dim=1)        # (B, N, D)
        logits = self.base.lm_head(hidden)           # (B, N, vocab)
 
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
 
        return CausalLMOutputWithPast(loss=loss, logits=logits)
 
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 100, **kw):
        """Minimal greedy generation (wraps base model for compatibility)."""
        self.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                out    = self.forward(input_ids)
                next_t = out.logits[:, -1, :].argmax(-1, keepdim=True)
                input_ids = torch.cat([input_ids, next_t], dim=1)
                if next_t.item() == self.base.config.eos_token_id:
                    break
        return input_ids

In [7]:
# ─────────────────────────────────────────────
# Factory
# ─────────────────────────────────────────────
 
def build_titans_mac(
    model_name: str = "Qwen/Qwen2-1.5B",
    cfg: Optional[TitansConfig] = None,
    dtype: torch.dtype = torch.bfloat16,
) -> tuple[Qwen2TitansMAC, object]:
    """Return (model, tokenizer)."""
    from transformers import AutoTokenizer
    if cfg is None:
        cfg = TitansConfig()
    base  = Qwen2ForCausalLM.from_pretrained(model_name, dtype=dtype)
    model = Qwen2TitansMAC(base, cfg).to(dtype)
    tok   = AutoTokenizer.from_pretrained(model_name)
    return model, tok
 

In [9]:

# ─────────────────────────────────────────────
# Training entry point
# ─────────────────────────────────────────────
 
cfg = TitansConfig(
    chunk_size=64,
    n_persistent=16,
    memory_depth=2,
    use_linear_memory=True,   # start with linear, then switch to MLP
    freeze_base=False,        # full fine-tune; set True for param-efficient
    inner_lr=0.01,
)

model, tokenizer = build_titans_mac("Qwen/Qwen2.5-1.5B-Instruct", cfg)

n_all = sum(p.numel() for p in model.parameters())
n_new = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {n_all:,}")
print(f"Trainable params: {n_new:,}")

# Optimizer (paper: AdamW, lr=4e-4, cosine, weight_decay=0.1)
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=4e-4,
    weight_decay=0.1,
)

# Quick smoke test
# tok_out = tokenizer("The capital of Italy is", return_tensors="pt")
# ids     = tok_out["input_ids"]
# out     = model(input_ids=ids, labels=ids)
# print(f"Loss: {out.loss.item():.4f}")
# out.loss.backward()
# opt.step()
# print("Backward pass OK.")
 
# For real training use HuggingFace Trainer or a custom loop with:
#   - Cosine LR schedule
#   - Gradient clipping (max_norm=1.0)
#   - Batch size ~0.5M tokens (per paper)
#   - Dataset: FineWeb-Edu or similar
 

Total params:     1,553,186,819
Trainable params: 1,553,186,819


In [ ]:
# ─────────────────────────────────────────────
# Training loop (two-phase) with Hugging Face dataset
# ─────────────────────────────────────────────


class SimpleTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length: int = 128):
        enc = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc.get("attention_mask", None)

    def __len__(self) -> int:
        return self.input_ids.shape[0]

    def __getitem__(self, idx: int) -> dict:
        item = {"input_ids": self.input_ids[idx]}
        if self.attention_mask is not None:
            item["attention_mask"] = self.attention_mask[idx]
        return item

def make_labels(input_ids: torch.Tensor, pad_token_id: Optional[int]) -> torch.Tensor:
    labels = input_ids.clone()
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100
    return labels

def set_base_trainable(model: Qwen2TitansMAC, trainable: bool):
    for p in model.base.parameters():
        p.requires_grad_(trainable)

def get_optimizer(model: Qwen2TitansMAC, lr: float):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=0.1,
    )

def train_phase(
    model: Qwen2TitansMAC,
    loader: DataLoader,
    epochs: int,
    lr: float,
    warmup_steps: int = 50,
    max_grad_norm: float = 1.0,
    device: Optional[torch.device] = None,
    pad_token_id: Optional[int] = None,
 ):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    opt = get_optimizer(model, lr)
    total_steps = max(1, epochs * len(loader))
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=min(warmup_steps, max(1, total_steps // 10)),
        num_training_steps=total_steps,
    )
    model.train()
    for epoch in range(epochs):
        running = 0.0
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)
            labels = make_labels(input_ids, pad_token_id)
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
            sched.step()
            opt.zero_grad()
            running += loss.item()
        print(f"epoch {epoch + 1}/{epochs} loss={running / len(loader):.4f}")

# Load a small subset from Hugging Face for a quick run.
# Replace with your target dataset when ready.
hf_ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")
texts = [t for t in hf_ds["text"] if t and t.strip()]

# Ensure padding exists for batching.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = SimpleTextDataset(texts, tokenizer, max_length=128)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

device = torch.device(device)
if device.type == "cpu":
    model = model.to(torch.float32)
model = model.to(device)

# Phase 1: linear memory + frozen base (train only new modules).
print("Phase 1: Training with linear memory and frozen base...")
set_base_trainable(model, False)
train_phase(
    model,
    loader,
    epochs=1,
    lr=4e-4,
    device=device,
    pad_token_id=tokenizer.pad_token_id,
 )
print("Phase 1 complete.")
print("Phase 2: Full fine-tuning (unfreeze base)...")
# Phase 2: full fine-tune (unfreeze base).
set_base_trainable(model, True)
train_phase(
    model,
    loader,
    epochs=1,
    lr=4e-4,
    device=device,
    pad_token_id=tokenizer.pad_token_id,
)
print("Phase 2 complete.")

# Note: for MLP memory set use_linear_memory=False before building the model
# (requires PyTorch >= 2.0 for torch.func).

epoch 1/1 loss=4.0305


KeyboardInterrupt: 

: 